initial prototype for MLR model, engineering features, and doing train test split

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.metrics import root_mean_squared_error, mean_absolute_percentage_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
import pickle as pkl
from scipy import stats
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

In [2]:
# import the data
df = pd.read_csv('../data/processed/hourly_usage_cleaned.csv')

In [3]:
# assign multiindex
df.set_index(['datetime', 'client_id'], inplace=True)

In [4]:
# create lagged features
df['lag_1hr'] = df.groupby(level='client_id')['hourly_usage_kwh'].shift(1)
df['lag_2hr'] = df.groupby(level='client_id')['hourly_usage_kwh'].shift(2)
df['lag_6hr'] = df.groupby(level='client_id')['hourly_usage_kwh'].shift(6)
df['lag_1dy'] = df.groupby(level='client_id')['hourly_usage_kwh'].shift(24)
df['lag_1wk'] = df.groupby(level='client_id')['hourly_usage_kwh'].shift(7*24)


# fine to do before train test split as would have this at inference
df['rolling_mean_1dy'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24).mean())
df['rolling_mean_1wk'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24*7).mean())

# get the rolling max, min and standard deviation (for volatility)
df['rolling_max_1dy'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24).max())
df['rolling_max_1wk'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24*7).max())

df['rolling_min_1dy'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24).min())
df['rolling_min_1wk'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24*7).min())

df['rolling_std_1dy'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24).std())
df['rolling_std_1wk'] = df.groupby(level='client_id')['hourly_usage_kwh'].transform(lambda x: x.rolling(window=24*7).std())

In [5]:
# and now get features for the hour, day of week and month. using cyclical encoding, 
# to avoid misinterpration of values (e.g., 1 and 12 for months are far numerically but similar with 
# regards to being winter)
df['hour'] = np.sin(2*np.pi*pd.to_datetime(df.index.get_level_values('datetime')).hour/24)
df['day_of_week'] = np.sin(2*np.pi*pd.to_datetime(df.index.get_level_values('datetime')).dayofweek/7)
df['month'] = np.sin(2*np.pi*pd.to_datetime(df.index.get_level_values('datetime')).month/12)


In [6]:
# remove nans where no feature
df.dropna(inplace=True)

df.xs(4, level='client_id').tail(50)

,hourly_usage_kwh,lag_1hr,lag_2hr,lag_6hr,lag_1dy,lag_1wk,rolling_mean_1dy,rolling_mean_1wk,rolling_max_1dy,rolling_max_1wk,rolling_min_1dy,rolling_min_1wk,rolling_std_1dy,rolling_std_1wk,hour,day_of_week,month
datetime,,,,,,,,,,,,,,,,,
2014-12-29 23:00:00,205.284553,234.247967,223.069106,110.264228,171.747967,203.760163,145.748645,146.528988,238.821138,251.524390,103.150407,71.646341,45.727452,40.907181,-2.588190e-01,0.000000,-2.449294e-16
2014-12-30 00:00:00,176.321138,205.284553,234.247967,174.796748,150.406504,173.780488,146.828421,146.544111,238.821138,251.524390,103.150407,71.646341,46.146269,40.917784,0.000000e+00,0.781831,-2.449294e-16
2014-12-30 01:00:00,154.471545,176.321138,205.284553,236.280488,132.621951,146.849593,147.738821,146.589479,238.821138,251.524390,103.150407,71.646341,46.069276,40.922350,2.588190e-01,0.781831,-2.449294e-16
2014-12-30 02:00:00,137.195122,154.471545,176.321138,238.821138,121.443089,122.459350,148.395156,146.677192,238.821138,251.524390,103.150407,71.646341,45.789720,40.886096,5.000000e-01,0.781831,-2.449294e-16
2014-12-30 03:00:00,121.443089,137.195122,154.471545,223.069106,116.361789,110.772358,148.606877,146.740708,238.821138,251.524390,103.150407,71.646341,45.646689,40.838245,7.071068e-01,0.781831,-2.449294e-16
2014-12-30 04:00:00,116.361789,121.443089,137.195122,234.247967,103.150407,108.739837,149.157351,146.786077,238.821138,251.524390,108.231707,71.646341,45.151652,40.799991,8.660254e-01,0.781831,-2.449294e-16
2014-12-30 05:00:00,111.788618,116.361789,121.443089,205.284553,108.739837,112.296748,149.284383,146.783053,238.821138,251.524390,108.231707,71.646341,45.037138,40.802582,9.659258e-01,0.781831,-2.449294e-16
2014-12-30 06:00:00,112.296748,111.788618,116.361789,176.321138,110.264228,109.247967,149.369072,146.801200,238.821138,251.524390,108.231707,71.646341,44.962423,40.786463,1.000000e+00,0.781831,-2.449294e-16
2014-12-30 07:00:00,128.556911,112.296748,111.788618,154.471545,123.475610,121.951220,149.580793,146.840520,238.821138,251.524390,108.231707,71.646341,44.847009,40.765542,9.659258e-01,0.781831,-2.449294e-16


In [7]:
# for now just do a super naive (non random split) of time series data - use earlier data for testing,
# and latter for training (use Time Series Split afterwards). 

# just filter using datetime directly

# define train proportion
train_proportion = 0.65
# first get timestamps for start and end time
start_date = pd.to_datetime(df.index.get_level_values('datetime')).min()
end_date = pd.to_datetime(df.index.get_level_values('datetime')).max()
# then the total time elapsed timedelta
tot_time_elasped = end_date - start_date
# multiply by train proportion and add to start date to get cutoff point
cutoff_date = start_date + tot_time_elasped*train_proportion

# now use cutoff data to filter for test and train
df_train = df[pd.to_datetime(df.index.get_level_values('datetime')) < cutoff_date]
df_test = df[pd.to_datetime(df.index.get_level_values('datetime')) > cutoff_date]

print('\n train shape: ', df_train.shape)
print('\n test shape: ', df_test.shape)

# now split into target and features (train and validation) (#TODO: do proper hyperparameter grid search
# for the different type of models)
y_train = df_train['hourly_usage_kwh']
y_validate = df_test['hourly_usage_kwh']

X_train = df_train.drop(columns=['hourly_usage_kwh'])
X_validate = df_test.drop(columns=['hourly_usage_kwh'])



# delete redundant dataframes
del df_train
del df_test
del df

#NOTE: need to scale features before input into linear regression - to stop large dominating!

# need for stratification by client? or maybe not because have all the features on per client basis already
# should be able to get back


 train shape:  (5833986, 17)

 test shape:  (4297202, 17)


In [8]:
# save the training and validation data 

with open('../data/processed/X_train.csv', 'wb') as f:
    pkl.dump(X_train, f)

with open('../data/processed/y_train.csv', 'wb') as f:
    pkl.dump(y_train, f)

with open('../data/processed/X_validate.csv', 'wb') as f:
    pkl.dump(X_validate, f)

with open('../data/processed/y_validate.csv', 'wb') as f:
    pkl.dump(y_validate, f)


In [ ]:
# try normalising data on a per client basis (will incorporate into an sklearn pipeline), and create additional
# version of variable (rolling mean in week) scaled globally to encode magnitude

# set up the scalar
scaler_global = StandardScaler()

# going to need to use a for loop to first fit a transformer on training data, 
# then transform both the training and testing

# for each column in X_train, need to store the fitted scaler (although would need transform)
# X_train_scalers = X_train.groupby(level='client_id').agg(lambda x: StandardScaler().fit(x.values.reshape(1, -1)))

# create empty list for scalars for each client (need to figure out datatype standardscaler to put in array)
scalers_per_client = []

# create empty list of dataframes, so csan append them
X_train_normalised_per_client = []
X_test_normalised_per_client = []

# get client_ids
# print(X_test.index.get_level_values('client_id').unique().difference(X_train.index.get_level_values('client_id')))

#NOTE: have clients in testing data that not the training data 
# (because of clean time split, and customer starting later)
# throws a spanner in the works for per client normalisation...

# # Now for each client_id, extract the appropriate scalar and use to transform both 
# for i in client_ids:
    
#     # get data for client for train and test
#     client_X_train = X_train.xs(i, level='client_id')
#     client_X_test = X_test.xs(i, level='client_id')
    
#     # fit standard scaler to client data
#     fitted_scaler = StandardScaler().fit(client_X_train)

#     # then use to transform training data, indexing into appropriate client
#     # X_train_normalised_per_client.xs((slice(None), i), level='client_id') = scalars_per_client[i].transform(X_train.xs(i, level='client_id'))
#     client_X_train_normalised = fitted_scaler.transform(client_X_train)

#     # and now also use to transform the testing data
#     # X_test_normalised_per_client.xs((slice(None), i), level='client_id') = scalars_per_client[i].transform(X_test.xs(i, level='client_id'))
#     client_X_test_normalised = fitted_scaler.transform(client_X_test)


#     print('\n client X_test: ', client_X_test.shape)
#     print('\n client X_test normalised: ', client_X_test_normalised.shape)
#     # # need to convert to dataframes because scaling outputs an array
#     client_X_train_normalised_df = pd.DataFrame(client_X_train_normalised, index=client_X_train.index, columns=client_X_train.columns)
#     client_X_test_normalised_df = pd.DataFrame(client_X_test_normalised, index=client_X_test.index, columns=client_X_test.columns)

#     print('\n client X_test normalised df: ', client_X_test_normalised_df.shape)
#     # and append to lists
#     scalers_per_client.append(fitted_scaler)
#     X_train_normalised_per_client.append(client_X_train_normalised_df)
#     X_test_normalised_per_client.append(client_X_test_normalised_df)

# # # check where size discrep occurs
# print(X_train.shape)
# print(sum(len(df) for df in X_train_normalised_per_client))

# now just concatenate list of dataframes.
# # because client id key was removed during .xs() extraction, add back in here
# X_train_normalised_per_client = pd.concat(X_train_normalised_per_client, keys=client_ids)
# X_test_normalised_per_client = pd.concat(X_test_normalised_per_client, keys=client_ids)

# now can use appropriate scaler somehow to transform scaling data
# X_train_normalised = X_train.groupby(level='client_id').apply(lambda x: )



# if using skleanr pipelines, would need to implement a custom transformer object to achieve per client normalisation

# easiest way may just be a column transformer (but doesn't save the mean and std needed)

# to get normalised target (rudimentary), can use transform with standardisation function. but need to reshape to 2D,
# for compatibility with scaler method. then need to ravel for compatibility with .transform()
# y_train_normalised  = y_train.groupby(level='client_id').transform(lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).ravel())
# y_test_normalised  = y_test.groupby(level='client_id').transform(lambda x: scaler.fit_transform(x.values.reshape(-1, 1)).ravel())

# X_train_normalised  = X_train.groupby(level='client_id').transform(lambda x: scaler.fit_transform(x))
# X_test_normalised  = X_test.groupby(level='client_id').transform(lambda x: scaler.fit_transform(x))

# TODO: get client-based normalisation for features
# then add global_scaled features to capture magnitude - use a rolling mean of past week (because want to keep context window the same)

# df_normalised_per_client = df.groupby(level='client_id').apply(lambda g: scaler.fit_transform(g))

Index([ 12,  39,  41, 106, 107, 108, 109, 110, 111, 112, 113, 115, 116, 117,
       120, 121, 122, 133, 160, 165, 178, 179, 181, 186, 289, 305, 322, 337],
      dtype='int64', name='client_id')


In [ ]:
# # save normalised features to csv
# X_train_normalised_per_client.to_csv('../data/processed/X_train_normalised_per_client.csv', index=False)
# X_test_normalised_per_client.to_csv('../data/processed/X_test_normalised_per_client.csv', index=False)

# # and scalars to pkl
# with open('../data/processed/scalars.pkl', 'wb') as f:
#     pkl.dump(scalers_per_client, f)

# del scalers_per_client

Try global normalisation for now

In [10]:
scaler_global = StandardScaler()
# fit the global scaler
scaler_global.fit(X_train)
# and use to transform both the training and testing data
X_train_normalised = pd.DataFrame(scaler_global.transform(X_train), index=X_train.index, columns=X_train.columns)

# save the global scaler, which will use on validation data
with open('../data/processed/scaler_global.pkl', 'wb') as f:
    pkl.dump(scaler_global, f)

# X_test_normalised = pd.DataFrame(scaler_global.transform(X_test), index=X_test.index, columns=X_test.columns)

In [44]:
X_train_normalised.head(50)

,,lag_1hr,lag_2hr,lag_6hr,lag_1dy,lag_1wk,rolling_mean_1dy,rolling_mean_1wk,rolling_max_1dy,rolling_max_1wk,rolling_min_1dy,rolling_min_1wk,rolling_std_1dy,rolling_std_1wk,hour,day_of_week,month
datetime,client_id,,,,,,,,,,,,,,,,
2012-01-08 01:00:00,1,-0.429025,-0.423253,-0.415570,-0.436622,-0.437716,-0.450356,-0.453460,-0.469863,-0.473056,-0.372840,-0.398642,-0.454550,-0.458008,0.365995,-1.106743,0.564922
2012-01-08 02:00:00,1,-0.435453,-0.429039,-0.421999,-0.436300,-0.437070,-0.450342,-0.453460,-0.469863,-0.473056,-0.372840,-0.398642,-0.454607,-0.458008,0.707089,-1.106743,0.564922
2012-01-08 03:00:00,1,-0.435774,-0.435467,-0.421999,-0.436622,-0.437716,-0.450328,-0.453458,-0.469863,-0.473056,-0.372840,-0.398642,-0.454666,-0.458016,0.999994,-1.106743,0.564922
2012-01-08 04:00:00,1,-0.436096,-0.435788,-0.430999,-0.436622,-0.437070,-0.450300,-0.453458,-0.469863,-0.473056,-0.372840,-0.398642,-0.454782,-0.458016,1.224747,-1.106743,0.564922
2012-01-08 05:00:00,1,-0.435774,-0.436110,-0.423285,-0.436300,-0.437393,-0.450285,-0.453456,-0.469863,-0.473056,-0.372840,-0.398642,-0.454839,-0.458023,1.366033,-1.106743,0.564922
2012-01-08 06:00:00,1,-0.435774,-0.435788,-0.429070,-0.436622,-0.437393,-0.450257,-0.453454,-0.469863,-0.473056,-0.372840,-0.398642,-0.454957,-0.458030,1.414223,-1.106743,0.564922
2012-01-08 07:00:00,1,-0.435774,-0.435788,-0.435499,-0.437265,-0.437070,-0.450215,-0.453456,-0.469863,-0.473056,-0.371309,-0.398642,-0.455147,-0.458023,1.366033,-1.106743,0.564922
2012-01-08 08:00:00,1,-0.436096,-0.435788,-0.435820,-0.435657,-0.437716,-0.450173,-0.453444,-0.469863,-0.473056,-0.371309,-0.398642,-0.455302,-0.458064,1.224747,-1.106743,0.564922
2012-01-08 09:00:00,1,-0.434489,-0.436110,-0.436142,-0.424400,-0.439330,-0.450201,-0.453361,-0.469863,-0.473056,-0.371309,-0.398642,-0.455360,-0.458173,0.999994,-1.106743,0.564922


In [ ]:
# for now could use Sklearn pipelines when dealing with sklearn models



# set up dictionary of models - for now just OLS and elastic net
model_dict = {
                'OLS': {'pipeline': Pipeline([('scale', StandardScaler()), ('clf', LinearRegression())]),
                        },
                'elastic_net': {'pipeline': Pipeline([('scale', StandardScaler()), ('clf', ElasticNet())]),
                                },
                
                'XGBoost': {Pipeline([('scale', StandardScaler()), ('clf', XGBRegressor())]),
                            },
              }

# # fit the models and predict using traininh data (no cross validation for now)
# for model_name, config in model_dict.items():

#     # if model_name != 'rgr_elastic':
#     #   continue
#     print('\n model: ', model_name)

#     # fit the model to training
#     config['model_object'].fit(X_train_normalised, y_train)

#     print('\n model fitted')


In [ ]:
# Do grid search with training data

# set up parameter grids
# dummy grid for OLS as no params to tune
model_dict['OLS']['param_grid'] = {'fit_intercept': True}

# for elastic net
model_dict['elastic_net']['param_grid']  = {'alpha': stats.loguniform(1e-2, 1e1),
                                            'l1_ratio': stats.uniform()}

# for XGBoost
model_dict['xgb']['param_grid'] = {'learning_rate': stats.loguniform(0.01, 1), 
                            'n_estimators': np.linspace(100, 2000, 10, dtype=np.int64),
                            'max_depth': np.linspace(2, 20, 10, dtype=np.int64), 
                            'gamma': np.linspace(0, 5, 6, dtype=np.float64),
                            'subsample': np.linspace(0.1, 1, 10, dtype=np.float64),
                            'colsample_bytree': np.linspace(0.1, 1, 10, dtype=np.float64),
                            'reg_alpha': np.linspace(0, 1, 11, dtype=np.float64),
                            'reg_lambda': np.linspace(0, 1, 11, dtype=np.float64),   
    }





# need to reorder training data, so that client Id is outer index. This way, when splitting 
# using number of rows (as time series split does), would then automatically 
X_train_normalised_reordered = X_train_normalised.sort_index(level='datetime')
y_train_reordered = y_train.sort_index(level='datetime')

# # X_train_normalised_reordered.head(50)
# y_train_reordered.head(50)
# set up time series splitter object
tscv = TimeSeriesSplit(n_splits=5)

# # TimeSeriesSplit(gap=0, max_train_size=None, n_splits=5, test_size=None)
# for i, (train_index, test_index) in enumerate(tscv.split(X_train_normalised)):
#     print(f"Fold {i}:")
#     print(f"  Train: index={train_index}")
#     print(f"  Train: index shape={train_index.shape}")
#     print(f"  Test:  index={test_index}")
#     print(f"  Test:  index shape={test_index.shape}")
for model_name, config in model_dict.items():

    print(f'\n model to fit: {model_name}')

    # set up the random search object
    rgr = RandomizedSearchCV(estimator=config['model_object'],
                              param_distributions=config['param_grid'], cv=tscv, 
                              scoring='neg_root_mean_squared_error', random_state=42)
    
    # fit the random search CV to the data
    rgr.fit(X=X_train_normalised_reordered, y=y_train_reordered)

    print(f'\n fitted')

    # save the model
    with open(f'../models/{model_name}.pkl', 'wb') as f:
        pkl.dump(rgr, f)

    print(f'\n saved')


c:\Users\Flynn\anaconda3\envs\dataProjects1\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=10. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(
c:\Users\Flynn\anaconda3\envs\dataProjects1\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.509e+09, tolerance: 1.029e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\Flynn\anaconda3\envs\dataProjects1\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.886e+09, tolerance: 1.658e+08
  model = cd_fast.enet_coordinate_descent(
c:

In [43]:
# import os
# for model_name in model_dict.keys():
#     size = os.path.getsize(f'../models/{model_name}.pkl')
#     print(f"{model_name}: {size} bytes")



# load in the models
for model_name, config in model_dict.items():

    if model_name == 'OLS':
        continue
    with open(f'../models/{model_name}.pkl', 'rb') as f:
            config['model_object'] = pkl.load(f)

In [56]:
# get coeffs of both models

df_coeffs = pd.DataFrame({'feature': model_dict['elastic_net']['model_object'].feature_names_in_, 
                        #   'coeff_lin_reg': model_dict['OLS']['model_object'].coef_,
                          'coeff_en': model_dict['elastic_net']['model_object'].best_estimator_.coef_,                      
                          })#.sort_values(by='coeff', key=abs)

pd.DataFrame(model_dict['elastic_net']['model_object'].cv_results_['params'])
# model_dict['elastic_net']['model_object'].best_index_
df_coeffs

,feature,coeff_en
0,lag_1hr,611.817676
1,lag_2hr,-131.048020
2,lag_6hr,-64.595112
3,lag_1dy,238.369009
4,lag_1wk,255.218893
5,rolling_mean_1dy,127.195433
6,rolling_mean_1wk,-129.332717
7,rolling_max_1dy,72.572417
8,rolling_max_1wk,-22.311029
9,rolling_min_1dy,44.159419


In [ ]:
# compute mean usage per client (in training set! 
# # probably fine but best to be consistent regarding data leakage)
# client_mean_usage = y_train.groupby(level='client_id').mean()


# # prediction loop

# for model_name, config in model_dict.items():
#     # run prediction (no cross validation for now)
#     y_pred = config['model_object'].predict(X_test_normalised)

#     print('\n prediction completed')


#     print('\n shape of y_pred: ', y_pred.shape)
#     print('\n shape of y_true: ', y_test.shape)
#     # compute rmse (try avoiding creation of dataframe at each timestep and just reference y_pred?)
#     config['rmse_per_client'] = pd.DataFrame({'y_true': y_test,
#                   'y_pred': y_pred}).groupby(level='client_id').apply(lambda g: root_mean_squared_error(y_true=g['y_true'], y_pred=g['y_pred']))

#     # get rmse normalised by the client mean usage
#     config['nrmse_per_client'] = config['rmse_per_client']/client_mean_usage

#     # and the mean nrmse
#     config['nrmse_per_mean'] = config['nrmse_per_client'].mean()

#     print('\n evalusation metrics done')



 prediction completed

 shape of y_pred:  (4297202,)

 shape of y_true:  (4297202,)

 evalusation metrics done

 prediction completed

 shape of y_pred:  (4297202,)

 shape of y_true:  (4297202,)

 evalusation metrics done

 prediction completed

 shape of y_pred:  (4297202,)

 shape of y_true:  (4297202,)

 evalusation metrics done

 prediction completed

 shape of y_pred:  (4297202,)

 shape of y_true:  (4297202,)

 evalusation metrics done


In [58]:
# X_train_normalised_reordered.describe().T

X_train.corr()

,lag_1hr,lag_2hr,lag_6hr,lag_1dy,lag_1wk,rolling_mean_1dy,rolling_mean_1wk,rolling_max_1dy,rolling_max_1wk,rolling_min_1dy,rolling_min_1wk,rolling_std_1dy,rolling_std_1wk,hour,day_of_week,month
lag_1hr,1.000000,0.991478,0.919683,0.983537,0.981634,0.950649,0.946909,0.941170,0.935328,0.898509,0.883864,0.785150,0.830252,-0.109921,0.001773,-0.024817
lag_2hr,0.991478,1.000000,0.939927,0.970444,0.968356,0.950939,0.946965,0.941248,0.935357,0.898635,0.883963,0.785218,0.830271,-0.111098,0.001516,-0.024861
lag_6hr,0.919683,0.939927,1.000000,0.896165,0.893498,0.951702,0.947115,0.941107,0.935349,0.898472,0.884205,0.785256,0.830243,-0.046834,0.000401,-0.024954
lag_1dy,0.983537,0.970444,0.896165,1.000000,0.984903,0.949832,0.947578,0.940233,0.935584,0.897689,0.884591,0.784336,0.830263,-0.101274,-0.002345,-0.025059
lag_1wk,0.981634,0.968356,0.893498,0.984903,1.000000,0.944409,0.946474,0.935757,0.934483,0.889816,0.883027,0.783486,0.830084,-0.101391,0.002300,-0.027316
rolling_mean_1dy,0.950649,0.950939,0.951702,0.949832,0.944409,1.000000,0.995578,0.988793,0.983070,0.944449,0.929456,0.824951,0.872523,-0.000020,-0.000547,-0.026255
rolling_mean_1wk,0.946909,0.946965,0.947115,0.947578,0.946474,0.995578,1.000000,0.986291,0.986930,0.936216,0.932677,0.827623,0.876197,-0.000018,0.000114,-0.027383
rolling_max_1dy,0.941170,0.941248,0.941107,0.940233,0.935757,0.988793,0.986291,1.000000,0.994235,0.893115,0.887877,0.892841,0.926333,-0.001295,-0.000883,-0.030144
rolling_max_1wk,0.935328,0.935357,0.935349,0.935584,0.934483,0.983070,0.986930,0.994235,1.000000,0.888945,0.885629,0.886484,0.929621,-0.000189,0.000124,-0.031593
rolling_min_1dy,0.898509,0.898635,0.898472,0.897689,0.889816,0.944449,0.936216,0.893115,0.888945,1.000000,0.955111,0.600771,0.682511,-0.000875,0.003565,-0.017195


In [91]:
for model_name, config in model_dict.items():
    
    print(f'\n mean nrmse for {model_name}: ', config['nrmse_per_mean'])

    # print('\n number of clients with nrmse below 0.1: ', config['nrmse_per_client'][config['nrmse_per_client'] < 0.1].shape)

    # print('\n number of clients with nrmse above 1: ', config['nrmse_per_client'][config['nrmse_per_client'] > 0.25].shape)

    print('\n clients above 1 nrmse: ', config['nrmse_per_client'][config['nrmse_per_client'] > 1.0])


 mean nrmse for rgr:  0.1143130702369497

 clients above 1 nrmse:  Series([], dtype: float64)

 mean nrmse for rgr_ridge:  0.11431291322163213

 clients above 1 nrmse:  Series([], dtype: float64)

 mean nrmse for rgr_lasso:  0.11358616979143099

 clients above 1 nrmse:  Series([], dtype: float64)

 mean nrmse for rgr_elastic:  0.11358616979143099

 clients above 1 nrmse:  Series([], dtype: float64)


In [90]:
# print(np.squeeze(rgr.coef_.shape))
# # print(rgr.intercept_)
# # print(rgr.feature_names_in_)

# create quick df of coeffs and feature names in
df_coeffs = pd.DataFrame({'feature': model_dict['rgr']['model_object'].feature_names_in_, 
                          'coeff_lin_reg': model_dict['rgr']['model_object'].coef_,
                          'coeff_ridge': model_dict['rgr_ridge']['model_object'].coef_,
                          'coeff_lasso': model_dict['rgr_lasso']['model_object'].coef_,
                          'coeff_en': model_dict['rgr_elastic']['model_object'].coef_,                      
                          })#.sort_values(by='coeff', key=abs)

df_coeffs.head(50)
# interestingly, ridge and lasso drove the month to zero. However, may well be because of failure to converge.

,feature,coeff_lin_reg,coeff_ridge,coeff_lasso,coeff_en
0,lag_1hr,638.352926,638.339666,516.018639,516.018639
1,lag_2hr,-152.783812,-152.773036,-15.715497,-15.715497
2,lag_6hr,-62.649848,-62.651050,-72.007198,-72.007198
3,lag_1dy,233.255988,233.258414,259.455191,259.455191
4,lag_1wk,253.831339,253.832090,227.295261,227.295261
5,rolling_mean_1dy,162.308129,162.291819,27.995946,27.995946
6,rolling_mean_1wk,-148.935463,-148.924647,-0.000000,-0.000000
7,rolling_max_1dy,65.315593,65.319703,29.858380,29.858380
8,rolling_max_1wk,-14.885426,-14.889885,-0.000000,-0.000000
9,rolling_min_1dy,31.361448,31.367374,16.216990,16.216990


In [ ]:
# # check prediction of linear regresison only for now
# y_pred = rgr_ridge.predict(X_test)

# # create df for y_pred
# df_predictions = pd.DataFrame({'y_pred': y_pred, 
#                                'y_true': y_test})

# # get rmse (client by client)
# rmse_by_client = df_predictions.groupby(level='client_id').apply(lambda g: root_mean_squared_error(y_true=g['y_true'], y_pred=g['y_pred']))

In [ ]:
# get mean usage by client (total)
client_mean_usage = df.groupby(level='client_id')['hourly_usage_kwh'].mean()

# compute normalised rmse
nrmse_by_client = rmse_by_client/client_mean_usage

print('mean nrmse: ', nrmse_by_client.mean())
print('nrmse below 0.1: ', nrmse_by_client[nrmse_by_client < 0.1].shape)


mean nrmse:  164.7580806730764
nrmse below 0.1:  (0,)


In [ ]:
# # create quick df of coeffs and feature names in
# df_coeffs = pd.DataFrame({'feature': rgr.feature_names_in_, 
#                           'coeff_lin_reg': rgr.coef_,
#                           'coeff_ridge': rgr_ridge.coef_,
#                           'coeff_lasso': rgr_lasso.coef_,
#                           'coeff_en': rgr_elastic.coef_,                      
#                           })#.sort_values(by='coeff', key=abs)

# df_coeffs.head(50)

,feature,coeff_lin_reg,coeff_ridge,coeff_lasso,coeff_en
0,lag_1hr,65.669317,65.669217,54.861891,32.713592
1,lag_2hr,-12.614988,-12.614901,-0.000000,17.055100
2,lag_6hr,-10.471337,-10.471347,-9.996502,-9.367967
3,lag_1dy,44.726925,44.726933,46.770513,36.478653
4,lag_1wk,43.381046,43.381055,42.922996,36.191526
5,rolling_mean_1dy,15.273485,15.273454,2.093637,4.983043
6,rolling_mean_1wk,-11.688374,-11.688348,-2.092461,0.000000
7,rolling_max_1dy,9.881581,9.881580,8.412378,5.497066
8,rolling_max_1wk,6.571923,6.571911,0.000000,1.753253
9,rolling_min_1dy,-7.654243,-7.654225,-1.706637,0.000000
